In [ ]:
# Standard library: filesystem and path utilities.
import os
# Standard library: glob pattern matching for file paths.
import glob
# Standard library: object-oriented path handling.
from pathlib import Path
# LangChain: load documents from directories and from plain text files.
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from pydantic import BaseModel, Field
# LangChain: split long text into fixed-size character chunks.
from langchain_text_splitters import CharacterTextSplitter
# LangChain: Chroma vector store for embeddings.
from langchain_chroma import Chroma
# LangChain: Hugging Face sentence-transformers embeddings.
from langchain_huggingface import HuggingFaceEmbeddings
# Litellm: OpenAI API wrapper.
from litellm import completion
from tqdm import tqdm
import uuid
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document


# Load environment variables from .env (e.g. API keys).
from dotenv import load_dotenv

# LLM model identifier (used elsewhere; not used in this script).
MODEL = "gpt-4.1-nano"

# Path to the Chroma vector DB directory (project root / vector_db).
# In notebooks __file__ is not defined; use cwd (run notebook from project root).
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "vector_db").exists() else Path.cwd().parent
DB_NAME = str(PROJECT_ROOT / "vector_db")
# Path to the knowledge base directory (project root / knowledge-base).
KNOWLEDGE_BASE = str(PROJECT_ROOT / "knowledge-base")

# Embedding model used to turn text into vectors (all-MiniLM-L6-v2).
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Load .env and override any existing env vars.
load_dotenv(override=True)

AVERAGE_CHUNK_SIZE = 500





In [ ]:

def fetch_documents():
    # List all top-level folders under the knowledge base.
    folders = glob.glob(str(Path(KNOWLEDGE_BASE) / "*"))
    # Accumulate loaded documents.
    documents = []
    for folder in folders:
        # Use folder name as document type (e.g. "docs", "faq").
        doc_type = os.path.basename(folder)
        # Loader: all .md files under this folder, UTF-8 text loader.
        loader = DirectoryLoader(
            folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
        )
        # Load documents from this folder.
        folder_docs = loader.load()
        for doc in folder_docs:
            # Tag each doc with its folder (doc_type).
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)
    return documents


In [ ]:
documents = fetch_documents()

In [ ]:
def make_prompt(document):
    how_many = (len(document.page_content) // AVERAGE_CHUNK_SIZE) + 1
    type = document.metadata.get("doc_type")
    source = document.metadata.get("source")
    return f"""
    You take a document and you split the document into overlapping chunks for a KnowledgeBase.

    The document is from the shared drive of a company called Insurellm.
    The document is of type: {type}
    The document has been retrieved from: {source}

    A chatbot will use these chunks to answer questions about the company.
    You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
    This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
    There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

    For each chunk, you should provide a headline, a summary, and the original text of the chunk.
    Together your chunks should represent the entire document with overlap.

    Here is the document:

    {document.page_content}

    Respond with the chunks.
    """


In [ ]:
print(documents[0].page_content)

In [ ]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [ ]:
make_messages(documents[0])

In [ ]:

class Result(BaseModel):
    page_content: str
    metadata: dict
    id: str | None = None  # Chroma.from_documents expects .id on each doc

In [ ]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document.metadata.get("source"), "type": document.metadata.get("doc_type")}
        return Result(
            page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,
            metadata=metadata,
            id=str(uuid.uuid4()),
        )


class Chunks(BaseModel):
    chunks: list[Chunk]

In [ ]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [ ]:
process_document(documents[0])

In [ ]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [ ]:
chunks = create_chunks(documents)

In [ ]:
print(len(chunks))

In [ ]:
def create_embeddings(chunks):
    # If DB already exists, delete its collection so we rebuild from scratch.
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    # Build a new Chroma store from chunks using our embedding model.
    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    # Access Chroma's underlying collection for count and sample.
    collection = vectorstore._collection
    count = collection.count()

    # Get one embedding to report vector dimensions.
    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore

In [ ]:
vectorstore = create_embeddings(chunks)

## Answering User


## Rerank Chunks


In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [ ]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    return retriever.invoke(question, k=RETRIEVAL_K)

In [ ]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)
print(chunks)

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list[Document]]:
    """
    Answer the given question with RAG; return the answer and the context documents.
    """
    
    query = rewrite_query(question, history)
    docs = fetch_context(query)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content, docs


In [ ]:
answer_question("Who won the IIOTY award?", [])